# Kathmandu Airbnb ML Analysis — Data Audit & Preprocessing

This notebook audits the seven supplied CSV files and creates cleaned working copies.
**Raw source files are never overwritten.** The cleaned files retain the source labels and
include QA flags where records look unusual. No synthetic rows are generated.

> Important provenance note: the property and price files label their source as
> `Synthetic Sample Data`. The booking-cancellation file has a hotel-booking schema and
> is not established as Kathmandu Airbnb data. Use it only as a separately labelled
> experimental cancellation dataset unless its provenance is verified.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path(".")  # Place the original CSV files in this folder, or change this path.
OUT_DIR = Path("preprocessed_output")
OUT_DIR.mkdir(exist_ok=True)

files = {
    "properties": "properties (1).csv",
    "availability": "property_availability.csv",
    "prices": "price_observations.csv",
    "bookings": "booking_cancellations.csv",
    "tourism": "nepal_monthly_tourist_arrivals.csv",
    "reviews": "property_reviews.csv",
    "occupancy": "kathmandu_monthly_occupancy.csv",
}
raw = {name: pd.read_csv(DATA_DIR / filename) for name, filename in files.items()}
{key: value.shape for key, value in raw.items()}


## 1. Structural and missing-value audit

In [ ]:
audit_rows = []
for name, df in raw.items():
    audit_rows.append({
        "dataset": name, "rows": len(df), "columns": len(df.columns),
        "duplicate_full_rows": int(df.duplicated().sum()),
        "missing_cells": int(df.isna().sum().sum()),
        "all_empty_columns": ", ".join(df.columns[df.isna().all()].tolist())
    })
audit = pd.DataFrame(audit_rows)
display(audit)


## 2. Key uniqueness and property-ID relationships

In [ ]:
properties = raw["properties"]
availability = raw["availability"]
prices = raw["prices"]

print("Duplicate property IDs:", properties["property_id"].duplicated().sum())
print("Duplicate property/date availability rows:",
      availability.duplicated(["property_id", "date"]).sum())
print("Duplicate property/observation/stay price rows:",
      prices.duplicated(["property_id", "observation_date", "stay_date"]).sum())

pids = set(properties.property_id.astype(str))
aids = set(availability.property_id.astype(str))
prids = set(prices.property_id.astype(str))
print("Property IDs missing in availability:", sorted(pids - aids)[:20], "count =", len(pids-aids))
print("Availability IDs absent from properties:", sorted(aids - pids)[:20], "count =", len(aids-pids))
print("Property IDs missing in prices:", sorted(pids - prids)[:20], "count =", len(pids-prids))
print("Price IDs absent from properties:", sorted(prids - pids)[:20], "count =", len(prids-pids))


## 3. Date parsing and cleaned working copies

In [ ]:
clean = {}

# Properties
df = raw["properties"].copy()
if "listing_url" in df and df["listing_url"].isna().all():
    df = df.drop(columns=["listing_url"])
df["property_id"] = df["property_id"].astype("string").str.strip()
df["captured_at"] = pd.to_datetime(df["captured_at"], utc=True, errors="coerce")
clean["properties_clean.csv"] = df

# Availability
df = raw["availability"].copy()
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["captured_at"] = pd.to_datetime(df["captured_at"], utc=True, errors="coerce")
df["property_id"] = df["property_id"].astype("string").str.strip()
df["is_available"] = df["is_available"].astype("boolean")
df["price_should_be_used_for_quote"] = df["is_available"].fillna(False)
clean["availability_clean.csv"] = df

# Price observations: source format is DD-MM-YYYY.
df = raw["prices"].copy()
for col in ["observation_date", "stay_date"]:
    df[col] = pd.to_datetime(df[col], format="%d-%m-%Y", errors="coerce")
df["captured_at"] = pd.to_datetime(df["captured_at"], utc=True, errors="coerce")
df["property_id"] = df["property_id"].astype("string").str.strip()
if "source_url" in df and df["source_url"].isna().all():
    df = df.drop(columns=["source_url"])
clean["price_observations_clean.csv"] = df

# Booking records: retain unusual rows but make them visible with QA flags.
df = raw["bookings"].copy()
for col in ["booking_date", "arrival_date"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")
df["adr_is_negative"] = df["adr"] < 0
df["guest_count_is_zero"] = df[["adults", "children", "babies"]].sum(axis=1).eq(0)
q1, q3 = df["adr"].quantile([.25, .75])
iqr = q3 - q1
df["adr_iqr_outlier_flag"] = (df["adr"] < q1 - 1.5*iqr) | (df["adr"] > q3 + 1.5*iqr)
df["total_nights"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
clean["booking_cancellations_clean.csv"] = df

# National tourism arrivals
df = raw["tourism"].copy()
df["month_date"] = pd.to_datetime(df["month_date"], errors="coerce")
df = df.sort_values("month_date").reset_index(drop=True)
df["year_month"] = df["month_date"].dt.to_period("M").astype("string")
clean["nepal_monthly_tourist_arrivals_clean.csv"] = df

# Keep empty files as schema-only artifacts; do not invent observations.
clean["property_reviews_schema_only.csv"] = raw["reviews"].copy()
clean["kathmandu_monthly_occupancy_schema_only.csv"] = raw["occupancy"].copy()

for filename, frame in clean.items():
    frame.to_csv(OUT_DIR / filename, index=False)
print("Saved", len(clean), "cleaned/schema files to", OUT_DIR.resolve())


## 4. Review unusual values before modelling

In [ ]:
book = clean["booking_cancellations_clean.csv"]
print("Negative ADR rows:", int(book["adr_is_negative"].sum()))
print("Zero-guest rows:", int(book["guest_count_is_zero"].sum()))
print("ADR IQR-flagged rows:", int(book["adr_iqr_outlier_flag"].sum()))
print("Empty review records:", len(raw["reviews"]))
print("Empty occupancy records:", len(raw["occupancy"]))
display(book.loc[book["adr_is_negative"] | book["guest_count_is_zero"],
                 ["booking_id", "adults", "children", "babies", "adr",
                  "adr_is_negative", "guest_count_is_zero"]].head(20))


## 5. Modelling cautions and next steps

- **Recommendation:** property metadata and amenity text can support a content-based recommender.
- **Price prediction:** combine property attributes with price observations, but use a chronological split and avoid mixing synthetic observations with claims about real market prices.
- **Availability:** the supplied calendar has 90 days per property. It is a snapshot window, not a long historical occupancy series.
- **Cancellation:** treat the hotel-booking file as a separate benchmark/experimental dataset. Do not present it as Kathmandu Airbnb bookings without source verification.
- **Tourism:** national monthly arrivals are contextual data, not Kathmandu-only demand.
- **Sentiment and occupancy:** currently unavailable because their CSVs contain headers but no rows.

Do not drop outliers automatically. Inspect them, document decisions, and fit imputers/encoders/scalers on training data only to prevent data leakage.
